# Nhận dạng dược liệu: CeNN + MobileNetV2 cho webcam C270, CÓ ảnh ngoài (TCMP-300, NB-TCM-CHM)

**Cài đặt Kaggle trước khi chạy:** *Settings → Accelerator: GPU T4 x2 (hoặc P100)* và *Internet: On* (cần bật để clone repo và tải dữ liệu ngoài).

Đây là một trong **hai notebook so sánh**. Hai bản dùng chung mã, seed và cách chia val/test, chỉ khác dữ liệu train:
- `../../../Downloads/train_repo_only_kaggle.ipynb`: chỉ dùng ảnh trong repo GitHub.
- `train_external_kaggle.ipynb`: ảnh repo cộng ảnh ngoài (TCMP-300, NB-TCM-CHM), kèm các biện pháp chống lệch.

Vì val/test giống hệt nhau, có thể so trực tiếp `per_class_metrics.csv` của hai bản để xem ảnh ngoài có làm giảm recall của **các lớp không được thêm ảnh** hay không. Để notebook tự so sánh: lưu output của bản này thành Dataset rồi *Add Input* vào bản kia (mục 7).

Notebook tự clone repo `NhanDangDuocLieu` và tự ghi code vào `/kaggle/working`. Nếu có tập test webcam, upload `C270_TEST/<state>/<lớp>/*.jpg` thành Kaggle Dataset rồi *Add Input*; notebook sẽ tự tìm.

Các bổ sung so với `train.ipynb` gốc: khối **CeNN** trước MobileNetV2 (so sánh với baseline), loại ảnh trùng trước khi chia tập, chuyển WebP sang PNG, augmentation mô phỏng C270, fine-tune 2 giai đoạn, đánh giá trên val sạch / val giả webcam / test C270 thật, báo cáo theo từng lớp, và xuất model `.keras`, TFLite cùng template CeNN `.h`.

In [ ]:
# ===== Cấu hình chuẩn bị =====
DATA_VARIANT = "external"
REPO_URL = "https://github.com/NgHuy-akaSteph/NhanDangDuocLieu.git"

import os
import sys
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
CODE_DIR = WORK / "cenn_c270"
CODE_DIR.mkdir(parents=True, exist_ok=True)


def find_dir(name, roots=("/kaggle/input",)):
    for root in roots:
        if Path(root).is_dir():
            for p in sorted(Path(root).rglob(name)):
                if p.is_dir():
                    return p
    return None


IMAGE_DIR = find_dir("DuocLieu")
if IMAGE_DIR is None:
    repo = WORK / "NhanDangDuocLieu"
    if not repo.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    IMAGE_DIR = repo / "DuocLieu"
print("DuocLieu:", IMAGE_DIR)

In [ ]:
%%writefile /kaggle/working/cenn_c270/cenn.py
"""Lớp CeNN (Cellular Neural Network, mạng nơ-ron tế bào) rời rạc theo thời gian
cho Keras, và các phép tăng cường mô phỏng webcam Logitech C270.

Mô hình CeNN chuẩn (Chua & Yang, 1988), mỗi tế bào (i, j) có:
    dx/dt = -x + A * y + B * u + z
    y     = f(x) = 0.5 * (|x + 1| - |x - 1|)  (= clip(x, -1, 1))
trong đó A (feedback template) và B (control template) là các nhân 3x3 dùng
chung cho mọi tế bào, z là bias, u là ảnh đầu vào.

Rời rạc hóa Euler với bước h, lặp T bước:
    x[t+1] = x[t] + h * (-x[t] + A * y[t] + B * u + z)

Ở đây A, B, z được HỌC bằng backprop cùng với MobileNetV2. Khối CeNN đặt
trước backbone, nối tắt (residual) với hệ số gamma khởi tạo = 0, nên lúc bắt
đầu train mô hình hành xử y hệt MobileNetV2 gốc -> không phá trọng số ImageNet.
Sau khi train, các template A/B/z có thể xuất ra (export_templates) để nhúng
trên vi điều khiển / FPGA đúng tinh thần CeNN phần cứng.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import tensorflow as tf
import keras
from keras import layers

# ---------------------------------------------------------------------------
# Lớp CeNN
# ---------------------------------------------------------------------------


def _edge_b_template(in_ch: int) -> np.ndarray:
    """Template B phát hiện biên kinh điển (Laplacian) chia đều cho các kênh."""
    lap = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], np.float32)
    return np.repeat(lap[:, :, None], in_ch, axis=2) / (in_ch * 8.0)


@keras.saving.register_keras_serializable(package="cenn")
class CeNN(layers.Layer):
    """Discrete-time CeNN với template học được.

    Args:
        cells: số "lớp tế bào" (kênh trạng thái). 4-8 là đủ, càng ít càng dễ nhúng.
        steps: số bước lặp T.
        h: bước thời gian Euler (0 < h <= 1). h nhỏ -> ổn định hơn.
        kernel: kích thước template (3 = lân cận r=1 chuẩn của CeNN).
        init: "edge" khởi tạo tế bào 0 bằng template phát hiện biên kinh điển,
              các tế bào còn lại ngẫu nhiên nhỏ; "random" ngẫu nhiên tất cả.
        out_channels: số kênh đầu ra (3 để đưa vào MobileNetV2 pretrained).
    """

    def __init__(self, cells=6, steps=4, h=0.25, kernel=3, init="edge",
                 out_channels=3, **kwargs):
        super().__init__(**kwargs)
        self.cells, self.steps, self.h = int(cells), int(steps), float(h)
        self.kernel, self.init, self.out_channels = int(kernel), init, int(out_channels)

    def build(self, input_shape):
        in_ch = int(input_shape[-1])
        k, c = self.kernel, self.cells
        rnd = keras.initializers.RandomNormal(stddev=0.05)
        b0 = np.asarray(rnd((k, k, in_ch, c)), np.float32)
        a0 = np.asarray(rnd((k, k, c, c)), np.float32) * 0.5
        z0 = np.zeros((c,), np.float32)
        # Tự hồi tiếp ở tâm template A (giữ trạng thái, đặc trưng của CeNN)
        for i in range(c):
            a0[k // 2, k // 2, i, i] = 1.0
        if self.init == "edge" and k == 3:
            b0[:, :, :, 0] = _edge_b_template(in_ch) * 4.0
            a0[:, :, :, 0] = 0.0
            a0[1, 1, 0, 0] = 2.0
            z0[0] = -0.5
        self.B = self.add_weight(name="B", shape=(k, k, in_ch, c), initializer="zeros")
        self.A = self.add_weight(name="A", shape=(k, k, c, c), initializer="zeros")
        self.z = self.add_weight(name="z", shape=(c,), initializer="zeros")
        self.B.assign(b0)
        self.A.assign(a0)
        self.z.assign(z0)
        self.proj = self.add_weight(name="proj", shape=(1, 1, c, self.out_channels),
                                    initializer=keras.initializers.GlorotUniform())
        # ReZero: bắt đầu bằng đồng nhất (identity) -> không làm hỏng backbone
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros")
        super().build(input_shape)

    def call(self, u):
        bu = tf.nn.conv2d(u, self.B, 1, "SAME") + self.z  # phần điều khiển: tính 1 lần
        x = bu
        for _ in range(self.steps):
            y = tf.clip_by_value(x, -1.0, 1.0)
            x = x + self.h * (-x + tf.nn.conv2d(y, self.A, 1, "SAME") + bu)
        y = tf.clip_by_value(x, -1.0, 1.0)
        out = tf.nn.conv2d(y, self.proj, 1, "SAME")
        if self.out_channels == u.shape[-1]:
            return tf.clip_by_value(u + self.gamma * out, -1.0, 1.0)
        return out

    def cell_outputs(self, u):
        """Trả về y (B, H, W, cells) để trực quan hóa bản đồ tế bào."""
        bu = tf.nn.conv2d(u, self.B, 1, "SAME") + self.z
        x = bu
        for _ in range(self.steps):
            y = tf.clip_by_value(x, -1.0, 1.0)
            x = x + self.h * (-x + tf.nn.conv2d(y, self.A, 1, "SAME") + bu)
        return tf.clip_by_value(x, -1.0, 1.0)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(cells=self.cells, steps=self.steps, h=self.h, kernel=self.kernel,
                   init=self.init, out_channels=self.out_channels)
        return cfg


def export_templates(cenn_layer: CeNN, out_dir: str | Path, name="cenn") -> dict:
    """Xuất A, B, z, proj, gamma ra JSON và header C (float) để nhúng."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    w = {v.name.split("/")[-1].split(":")[0]: v.numpy() for v in cenn_layer.weights}
    meta = dict(cells=cenn_layer.cells, steps=cenn_layer.steps, h=cenn_layer.h,
                kernel=cenn_layer.kernel, layout="A,B: [kh][kw][in][out]")
    (out_dir / f"{name}_templates.json").write_text(
        json.dumps({**meta, **{k: v.tolist() for k, v in w.items()}}, indent=1), encoding="utf-8")
    lines = [f"// CeNN templates - sinh tự động. {meta['layout']}",
             f"#pragma once",
             f"#define CENN_CELLS {meta['cells']}",
             f"#define CENN_STEPS {meta['steps']}",
             f"#define CENN_KERNEL {meta['kernel']}",
             f"static const float CENN_H = {meta['h']}f;"]
    for k, v in w.items():
        flat = ", ".join(f"{x:.7g}f" for x in np.ravel(v))
        dims = "".join(f"[{d}]" for d in v.shape) if v.ndim else ""
        lines.append(f"static const float CENN_{k.upper()}{dims if dims else ''} = "
                     f"{'{' + flat + '}' if v.ndim else flat};")
    (out_dir / f"{name}_templates.h").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return w


# ---------------------------------------------------------------------------
# Tăng cường mô phỏng webcam C270 (720p, fixed focus, nén MJPEG, nhiễu low-light)
# Áp dụng trong tf.data cho từng ảnh (chưa batch), giá trị pixel 0..255 float32.
# ---------------------------------------------------------------------------


def _gaussian_kernel(sigma, size=5):
    r = tf.range(size, dtype=tf.float32) - (size - 1) / 2.0
    g = tf.exp(-(r ** 2) / (2.0 * sigma ** 2))
    g = g / tf.reduce_sum(g)
    k2 = tf.tensordot(g, g, axes=0)
    return tf.tile(k2[:, :, None, None], [1, 1, 3, 1])


def webcam_degrade(image, strength=1.0, seed=None):
    """Mô phỏng ảnh chụp bằng C270. image: (H, W, 3) float32 0..255.

    seed=None -> ngẫu nhiên mỗi lần (dùng khi train).
    seed=tensor int shape [2] -> tất định (dùng tạo tập val "giả webcam" cố định).
    """
    s = float(strength)
    shape = tf.shape(image)
    h, w = shape[0], shape[1]
    counter = [0]

    def _seed():
        counter[0] += 1
        return tf.cast(seed, tf.int32) + tf.constant([0, 7919], tf.int32) * counter[0]

    def u(lo, hi):
        if seed is None:
            return tf.random.uniform([], lo, hi)
        return tf.random.stateless_uniform([], _seed(), lo, hi)

    class _Rng:
        @staticmethod
        def normal(shp, stddev=1.0):
            if seed is None:
                return tf.random.normal(shp, stddev=stddev)
            return tf.random.stateless_normal(shp, _seed(), stddev=stddev)

    rng = _Rng()

    # 1) Phân giải thấp / lấy nét cố định: thu nhỏ rồi phóng lại
    if s > 0:
        f = u(1.0 - 0.6 * s, 1.0)
        small = tf.cast(tf.cast(tf.stack([h, w]), tf.float32) * f, tf.int32)
        small = tf.maximum(small, 16)
        image = tf.image.resize(tf.image.resize(image, small, antialias=True), [h, w])

    # 2) Mờ (defocus) với xác suất 0.35
    def blur(img):
        k = _gaussian_kernel(u(0.4, 0.4 + 1.2 * s))
        return tf.nn.depthwise_conv2d(img[None], k, [1, 1, 1, 1], "SAME")[0]
    image = tf.cond(u(0.0, 1.0) < 0.35 * s, lambda: blur(image), lambda: image)

    # 3) Cân bằng trắng tự động lệch + phơi sáng
    temp = u(-0.15 * s, 0.15 * s)
    gains = tf.stack([1.0 + temp, 1.0 + u(-0.04, 0.04) * s, 1.0 - temp])
    exposure = u(1.0 - 0.4 * s, 1.0 + 0.15 * s)
    image = image * gains * exposure

    # 4) Nhiễu cảm biến, mạnh hơn khi tối
    sigma = u(0.0, 8.0 * s) * (1.5 - 0.5 * tf.minimum(exposure, 1.0))
    image = image + rng.normal(tf.shape(image), stddev=1.0) * sigma
    image = tf.clip_by_value(image, 0.0, 255.0)

    # 5) Nén JPEG/MJPEG
    def jpeg(img):
        q = tf.cast(u(40.0, 92.0), tf.int32)
        return tf.image.adjust_jpeg_quality(img / 255.0, q) * 255.0
    image = tf.cond(u(0.0, 1.0) < 0.7 * s, lambda: jpeg(image), lambda: image)
    image = tf.ensure_shape(image, [None, None, 3])
    return tf.clip_by_value(image, 0.0, 255.0)


# ---------------------------------------------------------------------------
# Mô hình
# ---------------------------------------------------------------------------


def build_model(num_classes, image_size, state="state", use_cenn=True, cenn_kwargs=None,
                mobilenet_alpha=1.0, dropout=0.25, lr=1e-3, weights="imagenet"):
    cenn_kwargs = cenn_kwargs or {}
    aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ], name="geometric_augmentation")

    base = keras.applications.MobileNetV2(
        input_shape=(image_size, image_size, 3), alpha=mobilenet_alpha,
        include_top=False, weights=weights)
    base.trainable = False

    inputs = keras.Input((image_size, image_size, 3), name="image")  # pixel 0..255
    x = aug(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="to_minus1_1")(x)  # = preprocess_input
    if use_cenn:
        x = CeNN(name="cenn", **cenn_kwargs)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)
    tag = "cenn_mobilenetv2" if use_cenn else "mobilenetv2"
    model = keras.Model(inputs, outputs, name=f"duoclieu_{state}_{image_size}_{tag}")
    compile_model(model, num_classes, lr)
    return model


def compile_model(model, num_classes, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        weighted_metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(k=min(2, num_classes), name="top2_accuracy"),
        ],
    )


def unfreeze_top(model, n_layers=30, lr=1e-5):
    """Fine-tune n_layers tầng cuối của MobileNetV2, giữ BatchNorm đóng băng."""
    base = next(l for l in model.layers if l.name.startswith("mobilenetv2"))
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, model.output_shape[-1], lr)
    return model


def center_square(frame_rgb: np.ndarray, frac: float = 0.8) -> np.ndarray:
    """Cắt vùng vuông ở giữa khung hình webcam (người dùng đặt mẫu vào giữa)."""
    h, w = frame_rgb.shape[:2]
    side = int(min(h, w) * frac)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return frame_rgb[y0:y0 + side, x0:x0 + side]


def count_flops_note(image_size, cells, steps, kernel=3, in_ch=3):
    """Ước lượng số phép nhân-cộng (MAC) của khối CeNN, phục vụ báo cáo nhúng."""
    px = image_size * image_size
    macs = px * kernel * kernel * in_ch * cells  # B*u
    macs += steps * px * kernel * kernel * cells * cells  # A*y mỗi bước
    macs += px * cells * 3  # proj
    return macs


if __name__ == "__main__":  # kiểm tra nhanh
    m = build_model(5, 96, weights=None)
    m.summary()
    x = tf.random.uniform((2, 96, 96, 3), 0, 255)
    print(m(x).shape, "CeNN MACs @96:", f"{count_flops_note(96, 6, 4) / 1e6:.1f}M")
    print(webcam_degrade(tf.random.uniform((96, 128, 3), 0, 255)).shape)

In [ ]:
%%writefile /kaggle/working/cenn_c270/download_external.py
"""Tải và lọc dữ liệu ngoài cho bài toán nhận dạng dược liệu.

Nguồn (đều CC BY 4.0, cần trích dẫn khi công bố):
  * TCMP-300  - Zhang et al., Scientific Data 2025, doi:10.1038/s41597-025-05522-7
                figshare: doi:10.6084/m9.figshare.29432726 (file tar.gz ~12,8 GB)
  * NB-TCM-CHM - Mendeley Data, doi:10.17632/2kjmzjyrmd.3

TCMP-300 chỉ có một file nén lớn, nên script ĐỌC LUỒNG (stream) file tar.gz và
chỉ ghi ra đĩa ảnh thuộc các loài khớp với repo -> cần băng thông ~12,8 GB nhưng
chỉ tốn ~0,3 GB dung lượng đĩa. Chạy trên Colab sẽ nhanh hơn nhiều so với máy nhà.

Kết quả:
  <OUT>/<ten_lop_repo>/<nguon>/<state>/*.jpg
  <OUT>/external_manifest.csv   (đường dẫn, lớp, state, nguồn, sha1, license)

Ví dụ:
  python download_external.py --out ExternalData                # cả hai nguồn
  python download_external.py --out ExternalData --skip-tcmp    # chỉ NB-TCM-CHM (nhỏ)
  python download_external.py --out ExternalData --tcmp-tar tcmp-300-release.tar.gz
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import re
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

TCMP_TAR_URL = "https://ndownloader.figshare.com/files/55751081"
MENDELEY_API = "https://data.mendeley.com/public-api/datasets/2kjmzjyrmd"
MENDELEY_VERSION = 3
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# --------------------------------------------------------------------------
# Ánh xạ loài ngoài -> lớp trong repo.
# state:  "fresh"  -> ảnh cây/lá sống, dùng cho nhánh tươi
#         "dried"  -> ảnh vị thuốc khô, dùng cho nhánh khô
#         "plant"  -> ảnh cây sống của loài mà repo chỉ có ảnh khô; KHÔNG trộn
#                     thẳng vào nhánh khô (khác miền ảnh), chỉ dùng để pretrain
# approx: True nếu chỉ gần đúng (khác loài/khác bộ phận) -> notebook mặc định bỏ
# --------------------------------------------------------------------------
TCMP_MAP = {
    "Perilla frutescens": ("tia_to", "fresh", False),
    "Plantago asiatica": ("ma_de", "fresh", False),
    "Paederia foetida": ("la_mo_long", "fresh", False),
    # Bồ công anh VN thường là Lactuca indica, TCMP là Taraxacum mongolicum
    "Taraxacum mongolicum": ("bo_cong_anh", "fresh", True),
    "Dioscorea polystachya": ("hoai_son", "plant", False),
    "Eucommia ulmoides": ("do_trong_bac", "plant", False),
    "Gleditsia sinensis": ("gai_bo_ket", "plant", False),
    "Lycium chinense": ("cau_ky_tu", "plant", False),
    "Rehmannia glutinosa": ("thuc_dia", "plant", False),
    "Panax notoginseng": ("nu_tam_that_bac", "plant", True),
    # Huyết đằng: kiểm tra mẫu của bạn là loài nào rồi bật approx=False cho loài đúng
    "Sargentodoxa cuneata": ("huyet_dang", "plant", True),
    "Spatholobus suberectus": ("huyet_dang", "plant", True),
}

# (thư mục Mendeley, lớp repo, state, approx)
NBTCM_MAP = {
    "Lycii Fructus": ("cau_ky_tu", "dried", False),
    # Sơn tra (Crataegus) khác loài với táo mèo (Docynia indica)
    "Crataegi Fructus": ("tao_meo", "dried", True),
    # Chỉ đúng nếu thư mục `sa` là SA NHÂN, không phải SẢ
    "Amomi Fructus": ("sa", "dried", True),
}


def _norm(s: str) -> str:
    return re.sub(r"[^a-z]+", " ", s.lower()).strip()


TCMP_PATTERNS = [(_norm(k), v) for k, v in TCMP_MAP.items()]


def _get(url: str, retries: int = 4, timeout: int = 120) -> bytes:
    for i in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "herb-dataset-fetcher/1.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except Exception as e:  # noqa: BLE001
            if i == retries - 1:
                raise
            print(f"  retry {i + 1} {url}: {e}", file=sys.stderr)
            time.sleep(2 * (i + 1))
    raise RuntimeError("unreachable")


class Writer:
    def __init__(self, out: Path):
        self.out = out
        self.out.mkdir(parents=True, exist_ok=True)
        self.manifest = out / "external_manifest.csv"
        self.seen: set[str] = set()
        self.rows: list[dict] = []
        if self.manifest.exists():
            with self.manifest.open(encoding="utf-8") as f:
                for row in csv.DictReader(f):
                    self.rows.append(row)
                    self.seen.add(row["sha1"])

    def save(self, data: bytes, cls: str, source: str, state: str, approx: bool,
             orig: str, license_: str) -> bool:
        sha1 = hashlib.sha1(data).hexdigest()
        if sha1 in self.seen:
            return False
        ext = Path(orig).suffix.lower()
        if ext not in IMG_EXT:
            ext = ".jpg"
        d = self.out / cls / source / state
        d.mkdir(parents=True, exist_ok=True)
        p = d / f"{source}_{sha1[:16]}{ext}"
        p.write_bytes(data)
        self.seen.add(sha1)
        self.rows.append({
            "path": p.relative_to(self.out).as_posix(), "class": cls, "state": state,
            "source": source, "approx": int(approx), "sha1": sha1,
            "original": orig, "license": license_,
        })
        return True

    def flush(self):
        keys = ["path", "class", "state", "source", "approx", "sha1", "original", "license"]
        with self.manifest.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(self.rows)


class _Progress(io.RawIOBase):
    """Bọc luồng HTTP để in tiến độ khi đọc tar.gz."""

    def __init__(self, raw, total: int | None):
        self.raw, self.total, self.n, self.t0, self.last = raw, total, 0, time.time(), 0.0

    def readable(self):
        return True

    def readinto(self, b):
        chunk = self.raw.read(len(b))
        n = len(chunk)
        b[:n] = chunk
        self.n += n
        now = time.time()
        if now - self.last > 10:
            self.last = now
            mb = self.n / 2**20
            speed = mb / max(now - self.t0, 1e-6)
            tot = f"/{self.total / 2**20:.0f}" if self.total else ""
            print(f"  TCMP-300: {mb:.0f}{tot} MB ({speed:.1f} MB/s)", flush=True)
        return n


def match_tcmp(member_path: str):
    parts = [_norm(p) for p in Path(member_path).parts[:-1]]
    for pat, val in TCMP_PATTERNS:
        if any(pat in p for p in parts):
            return val
    return None


def fetch_tcmp(writer: Writer, tar_path: str | None):
    if tar_path:
        print(f"Đọc TCMP-300 từ file local {tar_path}")
        tf = tarfile.open(tar_path, mode="r|gz")
    else:
        print(f"Stream TCMP-300 từ {TCMP_TAR_URL} (~12,8 GB, chỉ lưu loài khớp)")
        req = urllib.request.Request(TCMP_TAR_URL, headers={"User-Agent": "herb-dataset-fetcher/1.0"})
        resp = urllib.request.urlopen(req, timeout=300)
        total = int(resp.headers.get("Content-Length") or 0) or None
        stream = io.BufferedReader(_Progress(resp, total), buffer_size=4 * 2**20)
        tf = tarfile.open(fileobj=stream, mode="r|gz")

    kept, dirs_seen = 0, set()
    for m in tf:
        if not m.isfile() or not m.name.lower().endswith(IMG_EXT):
            continue
        top = str(Path(m.name).parent)
        if top not in dirs_seen and len(dirs_seen) < 3:
            print(f"  ví dụ thư mục trong tar: {top}")
        dirs_seen.add(top)
        hit = match_tcmp(m.name)
        if not hit:
            continue
        f = tf.extractfile(m)
        if f is None:
            continue
        cls, state, approx = hit
        if writer.save(f.read(), cls, "tcmp300", state, approx, m.name, "CC BY 4.0"):
            kept += 1
            if kept % 200 == 0:
                writer.flush()
                print(f"  đã giữ {kept} ảnh TCMP-300")
    tf.close()
    writer.flush()
    print(f"TCMP-300: giữ {kept} ảnh từ {len(dirs_seen)} thư mục")


def fetch_nbtcm(writer: Writer):
    print("Tải NB-TCM-CHM (Mendeley)")
    folders = json.loads(_get(f"{MENDELEY_API}/folders/{MENDELEY_VERSION}"))
    names = {f["id"]: f["name"] for f in folders}
    kept = 0
    for fo in folders:
        if fo["name"] not in NBTCM_MAP:
            continue
        parent = names.get(fo.get("parent_id"), "")
        # Dataset 2 = ảnh điện thoại chụp tại nhà thuốc -> gần điều kiện webcam nhất
        source = "nbtcm_phone" if parent == "Dataset 2" else "nbtcm_web"
        cls, state, approx = NBTCM_MAP[fo["name"]]
        files = json.loads(_get(f"{MENDELEY_API}/files?folder_id={fo['id']}&version={MENDELEY_VERSION}"))
        print(f"  {parent}/{fo['name']}: {len(files)} file -> {cls}")
        for fi in files:
            if not fi["filename"].lower().endswith(IMG_EXT):
                continue
            sha = fi.get("content_details", {}).get("sha256_hash")
            data = _get(fi["content_details"]["download_url"])
            if sha and hashlib.sha256(data).hexdigest() != sha:
                print(f"  bỏ {fi['filename']}: sai sha256", file=sys.stderr)
                continue
            if writer.save(data, cls, source, state, approx, f"{parent}/{fo['name']}/{fi['filename']}", "CC BY 4.0"):
                kept += 1
        writer.flush()
    print(f"NB-TCM-CHM: giữ {kept} ảnh")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", default="ExternalData")
    ap.add_argument("--skip-tcmp", action="store_true")
    ap.add_argument("--skip-nbtcm", action="store_true")
    ap.add_argument("--tcmp-tar", help="Đường dẫn tcmp-300-release.tar.gz đã tải sẵn (nếu có)")
    a = ap.parse_args()
    w = Writer(Path(a.out))
    if not a.skip_nbtcm:
        fetch_nbtcm(w)
    if not a.skip_tcmp:
        fetch_tcmp(w, a.tcmp_tar)
    w.flush()
    print(f"Xong. Manifest: {w.manifest}")


if __name__ == "__main__":
    main()

In [ ]:
# ===== Tải dữ liệu ngoài vào /kaggle/working/ExternalData =====
DOWNLOAD_NBTCM = True    # ~40 ảnh quả khô (câu kỷ tử, ...), nhanh
DOWNLOAD_TCMP = True     # TCMP-300: đọc luồng 12,8 GB, chỉ giữ loài khớp (10-30 phút). False = chạy nhanh
# Nếu đã có ExternalData dạng Kaggle Dataset (lưu từ lần chạy trước) thì dùng luôn, khỏi tải lại
_ext_input = find_dir("ExternalData")
if _ext_input is not None and (_ext_input / "external_manifest.csv").exists():
    EXTERNAL_DIR = _ext_input
else:
    EXTERNAL_DIR = WORK / "ExternalData"
    args = [sys.executable, str(CODE_DIR / "download_external.py"), "--out", str(EXTERNAL_DIR)]
    if not DOWNLOAD_NBTCM:
        args.append("--skip-nbtcm")
    if not DOWNLOAD_TCMP:
        args.append("--skip-tcmp")
    if DOWNLOAD_NBTCM or DOWNLOAD_TCMP:
        # In tiến độ của tiến trình con ra ô notebook
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="")
        assert proc.wait() == 0, "Tải dữ liệu ngoài lỗi (kiểm tra Internet: On trong Settings)"
print("ExternalData:", EXTERNAL_DIR)

In [ ]:
import sys
import gc
import json
import hashlib
import random
from datetime import datetime

sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from PIL import Image
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support

import cenn
from cenn import CeNN, build_model, unfreeze_top, webcam_degrade, export_templates, count_flops_note

QUICK = os.environ.get("NB_QUICK") == "1"  # chạy thử nhanh (kiểm tra code), không dùng để báo cáo

SEED = 42
IMAGE_SIZES = (96,) if QUICK else (128, 160)
VARIANTS = {"baseline": False, "cenn": True}   # so sánh có / không CeNN
STATES = ("fresh", "dried")
ALPHA_THRESHOLD = 0.85
BATCH_SIZE = 16
EPOCHS_HEAD = 1 if QUICK else 30
EPOCHS_FINETUNE = 1 if QUICK else 15
FINE_TUNE_LAYERS = 30
MIN_IMAGES = 7
VAL_RATIO = 0.20
MIN_VAL_IMAGES = 2

CENN_KWARGS = dict(cells=6, steps=4, h=0.25, kernel=3, init="edge")

# Dữ liệu ngoài (chỉ có tác dụng ở bản external)
USE_EXTERNAL = True
INCLUDE_APPROX = False          # True: dùng cả loài chỉ gần đúng (sơn tra ~ táo mèo, ...)
USE_PLANT_FOR_DRIED = False     # True: thêm ảnh cây sống vào nhánh khô (khác miền ảnh, cân nhắc)
# --- Chống lệch ---
MAX_EXTERNAL_PER_CLASS = 150    # trần tuyệt đối
MAX_EXTERNAL_RATIO = 2.0        # và không vượt quá 2 lần số ảnh train của repo cho lớp đó
EXTERNAL_WEIGHT = 0.5           # trọng số mỗi ảnh ngoài (ảnh repo = 1)
EXTERNAL_WEBCAM_PROB = 1.0      # ảnh ngoài LUÔN bị làm nhiễu kiểu C270 -> xoá dấu hiệu "ảnh web"
EXTERNAL_STAGE1_ONLY = True     # giai đoạn fine-tune chỉ dùng ảnh repo -> chốt lại trên miền ảnh thật

# Các lớp có ảnh ngoài ở cấu hình mặc định của bản external. Hai notebook dùng CÙNG danh sách này
# để nhóm kết quả theo lớp "có thêm ảnh" / "không thêm ảnh" khi so sánh.
AUG_CLASSES = {"fresh": ["la_mo_long", "ma_de", "tia_to"], "dried": ["cau_ky_tu"]}

EXTERNAL_DIR = globals().get("EXTERNAL_DIR")  # chỉ bản external mới tải dữ liệu ngoài

# Augmentation webcam
WEBCAM_AUG_PROB = 0.8
WEBCAM_STRENGTH = 1.0
C270_ROI_FRAC = 0.8             # giống c270_tool.py: cắt vuông 80% ở giữa khung hình

TF_FORMATS = {"JPEG", "PNG", "BMP", "GIF"}
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

keras.utils.set_random_seed(SEED)

C270_TEST_DIR = find_dir("C270_TEST") or (WORK / "C270_TEST")
CONVERTED_DIR = WORK / "_converted_images"   # /kaggle/input chỉ đọc -> ảnh chuyển đổi ghi vào working

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + ("_quick" if QUICK else "")
RUN_DIR = WORK / "experiments_cenn" / f"{DATA_VARIANT}_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Biến thể dữ liệu:", DATA_VARIANT)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("External:", bool(EXTERNAL_DIR and (EXTERNAL_DIR / "external_manifest.csv").exists()),
      "| C270 test:", C270_TEST_DIR.is_dir())
print("Kết quả lưu tại:", RUN_DIR)

## 1. Quét ảnh, chuẩn hóa định dạng, loại trùng

In [ ]:
def file_md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def tf_readable_path(path):
    """Trả về đường dẫn TF đọc được. WebP/khác -> chuyển sang PNG trong CONVERTED_DIR."""
    try:
        with Image.open(path) as im:
            fmt = im.format
            if fmt in TF_FORMATS:
                im.verify()
                return str(path), fmt
            CONVERTED_DIR.mkdir(parents=True, exist_ok=True)
            out = CONVERTED_DIR / (hashlib.md5(str(path).encode()).hexdigest()[:16] + ".png")
            if not out.exists():
                im.convert("RGB").save(out)
            return str(out), f"{fmt}->PNG"
    except Exception as err:  # noqa: BLE001
        print(f"Ảnh lỗi, bỏ qua: {path} ({err})")
        return None, None


def scan_folder(folder):
    if not folder.is_dir():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONS)


rows = []
for species_dir in sorted(p for p in IMAGE_DIR.iterdir() if (p / "raw").is_dir()):
    for state in STATES:
        for p in scan_folder(species_dir / "raw" / state):
            readable, fmt = tf_readable_path(p)
            if readable:
                rows.append(dict(path=readable, original=str(p), class_name=species_dir.name,
                                 state=state, source="repo", fmt=fmt, md5=file_md5(p)))
repo_df = pd.DataFrame(rows)
print("Ảnh repo đọc được:", len(repo_df))
print(repo_df["fmt"].value_counts().to_string())

# Trùng hệt nhưng KHÁC lớp trong cùng trạng thái -> nhãn mâu thuẫn, loại hết
conflict = repo_df.groupby(["state", "md5"])["class_name"].nunique()
conflict_md5 = set(conflict[conflict > 1].index.get_level_values("md5"))
if conflict_md5:
    print("Ảnh trùng giữa các lớp khác nhau (loại):")
    print(repo_df[repo_df.md5.isin(conflict_md5)][["state", "class_name", "original"]].to_string())
repo_df = repo_df[~repo_df.md5.isin(conflict_md5)]

before = len(repo_df)
repo_df = repo_df.drop_duplicates(["state", "class_name", "md5"]).reset_index(drop=True)
print(f"Loại {before - len(repo_df)} ảnh trùng hệt trong cùng lớp -> còn {len(repo_df)}")

summary_df = repo_df.pivot_table(index="class_name", columns="state", values="path",
                                 aggfunc="count", fill_value=0)
summary_df.to_csv(RUN_DIR / "dataset_summary_dedup.csv")
summary_df.T

## 2. Dữ liệu ngoài (chỉ dùng cho train)

Xem `ExternalData/external_manifest.csv` để biết nguồn và license từng ảnh. Số ảnh ngoài mỗi lớp sẽ bị giới hạn thêm theo `MAX_EXTERNAL_RATIO` ở bước chia tập.

In [ ]:
ext_df = pd.DataFrame()
manifest_path = EXTERNAL_DIR / "external_manifest.csv"
if USE_EXTERNAL and manifest_path.exists():
    ext_df = pd.read_csv(manifest_path)
    ext_df["path"] = [str(EXTERNAL_DIR / p) for p in ext_df["path"]]
    if not INCLUDE_APPROX:
        ext_df = ext_df[ext_df["approx"] == 0]
    if USE_PLANT_FOR_DRIED:
        ext_df.loc[ext_df["state"] == "plant", "state"] = "dried"
    ext_df = ext_df[ext_df["state"].isin(STATES)].copy()

    conv = [tf_readable_path(p) for p in ext_df["path"]]
    ext_df["path"] = [c[0] for c in conv]
    ext_df["fmt"] = [c[1] for c in conv]
    ext_df = ext_df.dropna(subset=["path"])
    ext_df["md5"] = [file_md5(p) for p in ext_df["path"]]
    # Bỏ ảnh ngoài trùng với ảnh repo (tránh rò rỉ sang val)
    ext_df = ext_df[~ext_df["md5"].isin(set(repo_df["md5"]))]
    ext_df = ext_df.drop_duplicates("md5")
    ext_df = ext_df.rename(columns={"class": "class_name"}).reset_index(drop=True)
    if len(ext_df):
        display(ext_df.groupby(["state", "class_name", "source"]).size().unstack(fill_value=0))
        actual = {s: sorted(ext_df[ext_df.state == s].class_name.unique()) for s in STATES}
        if any(sorted(AUG_CLASSES.get(s, [])) != actual[s] for s in STATES):
            print("CHÚ Ý: lớp có ảnh ngoài khác AUG_CLASSES (do đổi cấu hình?).", actual,
                  "\nCập nhật AUG_CLASSES ở CẢ HAI notebook để so sánh đúng nhóm.")
    else:
        print("Không còn ảnh ngoài nào sau khi lọc.")
else:
    print("Không dùng dữ liệu ngoài (USE_EXTERNAL=False hoặc chưa có manifest).")

## 3. Chia tập

Val chỉ gồm ảnh của repo (đã loại trùng), nên giống hệt nhau ở cả hai notebook. Ảnh ngoài (nếu có) chỉ vào train. Tập `test_c270` lấy từ `C270_TEST/<state>/<lớp>/`.

In [ ]:
ext_df = globals().get("ext_df", pd.DataFrame())


def build_manifest(state):
    sdf = repo_df[repo_df.state == state]
    counts = sdf.groupby("class_name").size()
    class_names = sorted(counts[counts >= MIN_IMAGES].index)
    assert len(class_names) >= 2, f"Nhánh {state} cần >= 2 lớp đủ {MIN_IMAGES} ảnh"
    label_of = {c: i for i, c in enumerate(class_names)}
    records, n_train_repo = [], {}
    for c in class_names:
        paths = sdf[sdf.class_name == c]["path"].tolist()
        random.Random(SEED + sum(map(ord, f"{state}:{c}"))).shuffle(paths)
        n_val = min(max(MIN_VAL_IMAGES, round(len(paths) * VAL_RATIO)), len(paths) - 1)
        n_train_repo[c] = len(paths) - n_val
        for i, p in enumerate(paths):
            records.append(dict(path=p, class_name=c, label=label_of[c], state=state,
                                split="val" if i < n_val else "train", source="repo",
                                source_weight=1.0, crop=0, is_ext=0))
    if USE_EXTERNAL and len(ext_df):
        for c in class_names:
            e = ext_df[(ext_df.state == state) & (ext_df.class_name == c)]
            if e.empty:
                continue
            cap = min(MAX_EXTERNAL_PER_CLASS, int(np.ceil(MAX_EXTERNAL_RATIO * n_train_repo[c])))
            e = e.sample(min(len(e), cap), random_state=SEED)
            print(f"[{state}] {c}: {n_train_repo[c]} ảnh repo train + {len(e)} ảnh ngoài (trần {cap})")
            for _, r in e.iterrows():
                records.append(dict(path=r.path, class_name=c, label=label_of[c],
                                    state=state, split="train", source=r.source,
                                    source_weight=EXTERNAL_WEIGHT, crop=0, is_ext=1))
    test_dir = C270_TEST_DIR / state
    if test_dir.is_dir():
        for cdir in sorted(test_dir.iterdir()):
            if cdir.name not in label_of:
                if cdir.is_dir():
                    print(f"[{state}] bỏ lớp test {cdir.name}: không có trong train")
                continue
            for p in scan_folder(cdir):
                readable, _ = tf_readable_path(p)
                if readable:
                    records.append(dict(path=readable, class_name=cdir.name, label=label_of[cdir.name],
                                        state=state, split="test_c270", source="c270",
                                        source_weight=1.0, crop=1, is_ext=0))
    m = pd.DataFrame(records)
    out = RUN_DIR / "dataset" / state
    out.mkdir(parents=True, exist_ok=True)
    m.to_csv(out / "split_manifest.csv", index=False)
    (out / "class_names.json").write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8")
    return m, class_names


manifests, class_names_by_state = {}, {}
for state in STATES:
    manifests[state], class_names_by_state[state] = build_manifest(state)
    print(f"\n=== {state.upper()} ===")
    display(pd.crosstab([manifests[state].class_name], [manifests[state].split, manifests[state].source]))

## 4. Pipeline dữ liệu

Train: đọc → resize_with_pad → cache → **giả webcam** (xác suất `WEBCAM_AUG_PROB`) → batch. Augmentation hình học (lật, xoay, zoom, tương phản) nằm trong model.
`val_webcam`: cùng ảnh val nhưng giả webcam với seed cố định, để so sánh độ bền giữa các cấu hình.

In [ ]:
def load_image(path, crop, image_size):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])

    def center():  # ảnh C270 gốc 1280x720 -> cắt vùng vuông giữa như lúc chạy thật
        s = tf.shape(img)
        side = tf.cast(tf.cast(tf.minimum(s[0], s[1]), tf.float32) * C270_ROI_FRAC, tf.int32)
        return tf.image.crop_to_bounding_box(img, (s[0] - side) // 2, (s[1] - side) // 2, side, side)

    img = tf.cond(crop > 0, center, lambda: img)
    return tf.image.resize_with_pad(tf.cast(img, tf.float32), image_size, image_size)


def class_weights(manifest):
    tr = manifest[manifest.split == "train"]
    eff = tr.groupby("label")["source_weight"].sum()  # số ảnh "hiệu dụng"
    return {int(k): float(eff.sum() / (len(eff) * v)) for k, v in eff.items()}


def make_dataset(manifest, split, image_size, training=False, webcam_sim=False):
    rows = manifest[manifest.split == split]
    if rows.empty:
        return None
    cw = class_weights(manifest)
    w = (rows["label"].map(cw) * rows["source_weight"]).to_numpy("float32") if training \
        else np.ones(len(rows), "float32")
    # Xác suất làm nhiễu kiểu C270 cho từng ảnh: ảnh ngoài dùng EXTERNAL_WEBCAM_PROB
    aug_p = np.where(rows["is_ext"].to_numpy() == 1, EXTERNAL_WEBCAM_PROB, WEBCAM_AUG_PROB).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((rows.path.tolist(), rows.crop.to_numpy("int32"),
                                             rows.label.to_numpy("int32"), w,
                                             np.arange(len(rows), dtype="int32"), aug_p))
    ds = ds.map(lambda p, c, y, w, i, a: (load_image(p, c, image_size), y, w, i, a),
                num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(rows), seed=SEED, reshuffle_each_iteration=True)

        def aug(x, y, w, i, a):
            x = tf.cond(tf.random.uniform([]) < a,
                        lambda: webcam_degrade(x, WEBCAM_STRENGTH), lambda: x)
            return tf.ensure_shape(x, [image_size, image_size, 3]), y, w
        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    elif webcam_sim:
        ds = ds.map(lambda x, y, w, i, a: (tf.ensure_shape(
                        webcam_degrade(x, WEBCAM_STRENGTH, seed=tf.stack([i, SEED])),
                        [image_size, image_size, 3]), y, w),
                    num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda x, y, w, i, a: (x, y, w))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Xem trước: ảnh gốc vs giả webcam
st = "fresh"
sample = manifests[st][manifests[st].split == "train"].sample(4, random_state=SEED)
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for j, (_, r) in enumerate(sample.iterrows()):
    x = load_image(tf.constant(r.path), tf.constant(0), 224)
    ax[0, j].imshow(x.numpy().astype("uint8")); ax[0, j].set_title(r.class_name, fontsize=9)
    ax[1, j].imshow(webcam_degrade(x, WEBCAM_STRENGTH, seed=tf.constant([j, 1])).numpy().astype("uint8"))
    ax[1, j].set_title("giả C270", fontsize=9)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout(); plt.savefig(RUN_DIR / "webcam_aug_preview.png", dpi=120); plt.show()

## 5. Huấn luyện: 2 giai đoạn, mỗi (kích thước × trạng thái × biến thể)

In [ ]:
def predict_split(model, manifest, split, image_size, webcam_sim=False):
    ds = make_dataset(manifest, split, image_size, webcam_sim=webcam_sim)
    if ds is None:
        return None
    probs = model.predict(ds.map(lambda x, y, w: x), verbose=0)
    rows = manifest[manifest.split == split].reset_index(drop=True)
    out = rows[["path", "class_name", "label"]].copy()
    out["pred_label"] = probs.argmax(1)
    out["confidence"] = probs.max(1)
    out["correct"] = out.label == out.pred_label
    return out


def alpha_metrics(pred, alpha=ALPHA_THRESHOLD, prefix=""):
    if pred is None or pred.empty:
        return {}
    acc = pred.confidence >= alpha
    return {
        f"{prefix}n": len(pred),
        f"{prefix}accuracy": float(pred.correct.mean()),
        f"{prefix}macro_f1": float(f1_score(pred.label, pred.pred_label, average="macro", zero_division=0)),
        f"{prefix}local_coverage": float(acc.mean()),
        f"{prefix}selective_accuracy": float(pred[acc].correct.mean()) if acc.any() else np.nan,
        f"{prefix}wrong_accept_rate": float((acc & ~pred.correct).mean()),
    }


def per_class_rows(pred, names, state, split, image_size, variant):
    """Precision/recall/F1 và tỷ lệ chấp nhận sai ở α cho TỪNG lớp."""
    if pred is None or pred.empty:
        return []
    labels = list(range(len(names)))
    p, r, f, s = precision_recall_fscore_support(pred.label, pred.pred_label, labels=labels, zero_division=0)
    accepted = pred.confidence >= ALPHA_THRESHOLD
    out = []
    for k, name in enumerate(names):
        acc_k = accepted & (pred.pred_label == k)
        out.append(dict(data_variant=DATA_VARIANT, image_size=image_size, model=variant, state=state,
                        split=split, class_name=name, has_external=name in AUG_CLASSES.get(state, []),
                        support=int(s[k]), precision=float(p[k]), recall=float(r[k]), f1=float(f[k]),
                        accepted_as_class=int(acc_k.sum()),
                        wrong_accept_rate=float((acc_k & ~pred.correct).sum() / acc_k.sum()) if acc_k.any() else np.nan))
    return out


def exp_dir(image_size, state, variant):
    return RUN_DIR / f"img_{image_size}" / state / variant


def train_experiment(image_size, state, variant):
    keras.backend.clear_session(); gc.collect()
    keras.utils.set_random_seed(SEED)
    out = exp_dir(image_size, state, variant); out.mkdir(parents=True, exist_ok=True)
    m, names = manifests[state], class_names_by_state[state]
    train_ds = make_dataset(m, "train", image_size, training=True)
    val_ds = make_dataset(m, "val", image_size)

    model = build_model(len(names), image_size, state=state, use_cenn=VARIANTS[variant],
                        cenn_kwargs=CENN_KWARGS, lr=1e-3)

    def cbs(tag, patience):
        return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
                keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
                keras.callbacks.CSVLogger(str(out / f"history_{tag}.csv"))]

    print(f"\n===== {state.upper()} | {image_size}px | {variant} | giai đoạn 1 (head{'+CeNN' if VARIANTS[variant] else ''}) =====")
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=cbs("head", 6), verbose=2)
    best1 = min(h1.history["val_loss"])
    stage1_path = out / "stage1.weights.h5"
    model.save_weights(stage1_path)

    ft_train_ds = train_ds
    if EXTERNAL_STAGE1_ONLY and (m.is_ext == 1).any():
        # Fine-tune chỉ trên ảnh repo (trọng số lớp tính lại trên ảnh repo)
        ft_train_ds = make_dataset(m[m.is_ext == 0], "train", image_size, training=True)
        print("Giai đoạn 2 chỉ dùng ảnh repo (EXTERNAL_STAGE1_ONLY=True)")
    print(f"===== giai đoạn 2: fine-tune {FINE_TUNE_LAYERS} tầng cuối MobileNetV2 =====")
    unfreeze_top(model, FINE_TUNE_LAYERS, lr=1e-5)
    h2 = model.fit(ft_train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE, callbacks=cbs("finetune", 5), verbose=2)
    best2 = min(h2.history["val_loss"])
    if best2 > best1:
        print(f"Fine-tune không cải thiện val_loss ({best2:.4f} > {best1:.4f}) -> giữ trọng số giai đoạn 1")
        model.load_weights(stage1_path)
    model.save(out / "best_model.keras")

    preds = {s: predict_split(model, m, s, image_size, webcam_sim=(s == "val_webcam"))
             for s in ("val", "test_c270")}
    preds["val_webcam"] = predict_split(model, m, "val", image_size, webcam_sim=True)
    for s, p in preds.items():
        if p is not None:
            p.to_csv(out / f"pred_{s}.csv", index=False)
    result = dict(image_size=image_size, state=state, variant=variant,
                  val_loss=min(best1, best2), finetune_used=best2 <= best1,
                  params=model.count_params(),
                  model_mb=(out / "best_model.keras").stat().st_size / 2**20)
    pc = []
    for s, p in preds.items():
        result.update(alpha_metrics(p, prefix=f"{s}_"))
        pc += per_class_rows(p, names, state, s, image_size, variant)
    hist = pd.concat([pd.DataFrame(h1.history).assign(stage="head"),
                      pd.DataFrame(h2.history).assign(stage="finetune")], ignore_index=True)
    del model, train_ds, ft_train_ds, val_ds
    keras.backend.clear_session(); gc.collect()
    return hist, result, pc


histories, results, per_class = {}, [], []
for image_size in IMAGE_SIZES:
    for state in STATES:
        for variant in VARIANTS:
            hist, res, pc = train_experiment(image_size, state, variant)
            histories[(image_size, state, variant)] = hist
            results.append(res)
            per_class += pc
            pd.DataFrame(results).assign(data_variant=DATA_VARIANT).to_csv(RUN_DIR / "training_summary.csv", index=False)
            pd.DataFrame(per_class).to_csv(RUN_DIR / "per_class_metrics.csv", index=False)

summary = pd.DataFrame(results)
per_class_df = pd.DataFrame(per_class)
cols = ["image_size", "state", "variant", "val_accuracy", "val_macro_f1", "val_webcam_accuracy",
        "val_webcam_macro_f1", "test_c270_accuracy", "test_c270_macro_f1",
        "val_local_coverage", "val_wrong_accept_rate", "model_mb"]
display(summary[[c for c in cols if c in summary.columns]].round(3))

## 6. Đường học và so sánh CeNN vs baseline

In [ ]:
fig, axes = plt.subplots(len(STATES), 2, figsize=(14, 4.5 * len(STATES)), squeeze=False)
for i, state in enumerate(STATES):
    for (sz, st_, var), h in histories.items():
        if st_ != state:
            continue
        ls = "-" if var == "cenn" else "--"
        axes[i, 0].plot(h["val_loss"].values, ls, label=f"{sz}px {var}")
        axes[i, 1].plot(h["val_accuracy"].values, ls, label=f"{sz}px {var}")
    axes[i, 0].set_title(f"{state}: val loss (head rồi fine-tune)")
    axes[i, 1].set_title(f"{state}: val accuracy"); axes[i, 1].set_ylim(0, 1.05)
    for a in axes[i]:
        a.grid(alpha=0.3); a.legend(fontsize=8); a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(RUN_DIR / "learning_curves.png", dpi=140); plt.show()

metric_cols = [c for c in ["val_macro_f1", "val_webcam_macro_f1", "test_c270_macro_f1"] if c in summary]
comp = summary.pivot_table(index=["state", "image_size"], columns="variant", values=metric_cols)
comp.to_csv(RUN_DIR / "cenn_vs_baseline.csv")
display(comp.round(3))
print("Lưu ý: val nhỏ (vài chục ảnh) -> chênh lệch vài % có thể chỉ là nhiễu. "
      "Hãy chạy lại với vài SEED khác và ưu tiên kết quả trên test_c270.")

## 7. Kết quả theo từng lớp và so sánh với notebook còn lại

Nhóm lớp theo `has_external`, tức có nằm trong `AUG_CLASSES` hay không. Ở bản repo-only cột này vẫn được tính để hai bản ghép với nhau được.
**Dấu hiệu bị lệch:** ở bản external, recall trung bình của nhóm `has_external=False` giảm so với bản repo-only, hoặc `wrong_accept_rate` của nhóm `True` tăng.

Để so sánh tự động: mở output của notebook kia → *New Dataset*, rồi *Add Input* dataset đó vào notebook này. Ô dưới sẽ tự tìm `per_class_metrics.csv` của biến thể còn lại trong `/kaggle/input`.

In [ ]:
group_cols = ["data_variant", "image_size", "model", "state", "split", "has_external"]
by_group = (per_class_df.groupby(group_cols)
            .agg(n_classes=("class_name", "count"), mean_recall=("recall", "mean"),
                 mean_f1=("f1", "mean"), mean_wrong_accept=("wrong_accept_rate", "mean"))
            .reset_index())
by_group.to_csv(RUN_DIR / "per_class_by_group.csv", index=False)
display(by_group.round(3))

# Tìm kết quả của notebook còn lại (nếu đã Add Input)
others = []
if Path("/kaggle/input").is_dir():
    for f in Path("/kaggle/input").rglob("per_class_metrics.csv"):
        d = pd.read_csv(f)
        if "data_variant" in d and (d.data_variant != DATA_VARIANT).all():
            others.append(d)
            print("Đã tìm thấy kết quả biến thể", d.data_variant.iloc[0], "tại", f)
if others:
    both = pd.concat([per_class_df, pd.concat(others)], ignore_index=True)
    key_cols = ["image_size", "model", "state", "split", "class_name", "has_external"]
    wide = both.pivot_table(index=key_cols, columns="data_variant", values=["recall", "wrong_accept_rate"])
    if ("recall", "external") in wide and ("recall", "repo_only") in wide:
        wide[("recall", "delta_ext_minus_repo")] = wide[("recall", "external")] - wide[("recall", "repo_only")]
        wide.to_csv(RUN_DIR / "compare_external_vs_repo_per_class.csv")
        delta = (wide[("recall", "delta_ext_minus_repo")].rename("delta_recall").reset_index()
                 .groupby(["image_size", "model", "state", "split", "has_external"])["delta_recall"]
                 .agg(["mean", "min", "count"]).reset_index())
        display(delta.round(3))
        print("delta_recall < 0 ở nhóm has_external=False -> ảnh ngoài đang kéo các lớp khác về phía lớp được thêm ảnh.")
        sub = wide[("recall", "delta_ext_minus_repo")].reset_index()
        sub = sub[sub.split == "val"]
        if len(sub):
            sub.columns = [c if isinstance(c, str) else c[0] for c in sub.columns]
            sub = sub.rename(columns={"recall": "delta"})
            for (sz, mdl, st_), g in sub.groupby(["image_size", "model", "state"]):
                g = g.sort_values("delta")
                plt.figure(figsize=(8, 0.28 * len(g) + 1.2))
                plt.barh(g.class_name, g.delta, color=["#d9534f" if h else "#5b8def" for h in g.has_external])
                plt.axvline(0, color="k", lw=0.8)
                plt.title(f"Recall(external) - Recall(repo_only) | {st_} {sz}px {mdl} | đỏ = lớp có ảnh ngoài")
                plt.tight_layout(); plt.savefig(RUN_DIR / f"delta_recall_{st_}_{sz}_{mdl}.png", dpi=120); plt.show()
else:
    print("Chưa có kết quả của notebook còn lại trong /kaggle/input. Bỏ qua bước so sánh chéo.")

## 8. Chọn cấu hình, xem bản đồ tế bào CeNN, ma trận nhầm lẫn

In [ ]:
# Ưu tiên test C270 thật; nếu chưa có thì dùng val giả webcam
key = "test_c270_macro_f1" if "test_c270_macro_f1" in summary and summary["test_c270_macro_f1"].notna().any() \
    else "val_webcam_macro_f1"
best = summary.sort_values(key, ascending=False).groupby("state").head(1).set_index("state")
print("Tiêu chí chọn:", key)
display(best[["image_size", "variant", key]])

for state in STATES:
    r = best.loc[state]
    out = exp_dir(int(r.image_size), state, r.variant)
    model = keras.models.load_model(out / "best_model.keras")
    names = class_names_by_state[state]
    pred = pd.read_csv(out / "pred_val.csv")
    cm = confusion_matrix(pred.label, pred.pred_label, labels=range(len(names)))
    plt.figure(figsize=(0.45 * len(names) + 3, 0.45 * len(names) + 2))
    plt.imshow(cm, cmap="Blues"); plt.colorbar()
    plt.xticks(range(len(names)), names, rotation=90, fontsize=7); plt.yticks(range(len(names)), names, fontsize=7)
    plt.title(f"{state} | {int(r.image_size)}px | {r.variant} | val"); plt.tight_layout()
    plt.savefig(out / "confusion_val.png", dpi=130); plt.show()

    if r.variant == "cenn":
        layer = model.get_layer("cenn")
        p0 = manifests[state][manifests[state].split == "val"].iloc[0]
        x = load_image(tf.constant(p0.path), tf.constant(int(p0.crop)), int(r.image_size))
        u = x[None] / 127.5 - 1.0
        cells = layer.cell_outputs(u).numpy()[0]
        fig, ax = plt.subplots(1, cells.shape[-1] + 2, figsize=(2.2 * (cells.shape[-1] + 2), 2.6))
        ax[0].imshow(x.numpy().astype("uint8")); ax[0].set_title(p0.class_name, fontsize=8)
        for k in range(cells.shape[-1]):
            ax[k + 1].imshow(cells[..., k], cmap="gray", vmin=-1, vmax=1); ax[k + 1].set_title(f"tế bào {k}", fontsize=8)
        ax[-1].imshow(((layer(u).numpy()[0] + 1) * 127.5).clip(0, 255).astype("uint8"))
        ax[-1].set_title(f"đầu ra (gamma={float(layer.gamma.numpy()):.3f})", fontsize=8)
        for a in ax:
            a.axis("off")
        plt.tight_layout(); plt.savefig(out / "cenn_cells.png", dpi=130); plt.show()
        print(f"CeNN MACs ở {int(r.image_size)}px: "
              f"{count_flops_note(int(r.image_size), layer.cells, layer.steps) / 1e6:.1f} triệu")
    del model; keras.backend.clear_session()

## 9. Xuất model để triển khai (Keras, TFLite, template CeNN)

In [ ]:
EXPORT_DIR = RUN_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)
for state in STATES:
    r = best.loc[state]
    src = exp_dir(int(r.image_size), state, r.variant) / "best_model.keras"
    model = keras.models.load_model(src)
    d = EXPORT_DIR / state; d.mkdir(exist_ok=True)
    model.save(d / "model.keras")
    (d / "class_names.json").write_text(json.dumps(class_names_by_state[state], ensure_ascii=False, indent=2),
                                        encoding="utf-8")
    (d / "config.json").write_text(json.dumps(dict(image_size=int(r.image_size), variant=r.variant,
                                                   alpha=ALPHA_THRESHOLD, roi_frac=C270_ROI_FRAC,
                                                   cenn=CENN_KWARGS if r.variant == "cenn" else None),
                                              indent=2), encoding="utf-8")
    try:
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]  # lượng tử hóa động (int8 trọng số)
        (d / "model_dynamic_int8.tflite").write_bytes(conv.convert())
    except Exception as err:  # noqa: BLE001
        print(f"[{state}] TFLite lỗi: {err}")
    if r.variant == "cenn":
        export_templates(model.get_layer("cenn"), d)
    print(state, "->", sorted(p.name for p in d.iterdir()))
    del model; keras.backend.clear_session()

## 10. Dự đoán: ảnh file hoặc ảnh chụp từ C270

Chạy thời gian thực trên máy có webcam:
`python cenn_c270/c270_tool.py predict --model <export>/dried/model.keras --classes <export>/dried/class_names.json --size <image_size>`

In [ ]:
MODEL_CACHE = {}


def student_predict(image_path, state, from_c270=False):
    if state not in MODEL_CACHE:
        d = EXPORT_DIR / state
        MODEL_CACHE[state] = (keras.models.load_model(d / "model.keras"),
                              json.loads((d / "class_names.json").read_text(encoding="utf-8")),
                              json.loads((d / "config.json").read_text(encoding="utf-8")))
    model, names, cfg = MODEL_CACHE[state]
    x = load_image(tf.constant(str(image_path)), tf.constant(int(from_c270)), cfg["image_size"])
    p = model(x[None], training=False).numpy()[0]
    k = int(p.argmax())
    top3 = [(names[i], round(float(p[i]), 3)) for i in p.argsort()[::-1][:3]]
    return dict(state=state, predicted_class=names[k], confidence=float(p[k]), top3=top3,
                alpha=ALPHA_THRESHOLD,
                decision="accept_local" if p[k] >= ALPHA_THRESHOLD else "send_to_gemini")


demo = manifests["dried"][manifests["dried"].split == "val"].iloc[0]
print("Thật:", demo.class_name)
print(student_predict(demo.path, "dried"))